# 01 — Data Extraction & Dataset Split

**Purpose:** Download images from Pexels and split them into `dataset/train/` and `dataset/test/`.

All logic lives in `src/data_extraction.py` and `src/split_data.py`.  
This notebook is the single place to run both steps.

---
### Folder output after running this notebook
```
AuraVision/
├── extracted_data/          ← all downloaded training images
│   ├── Children/
│   ├── Adults/
│   └── Seniors/
└── dataset/          ← 80/20 split ready for training
    ├── train/
    └── test/
```

## Setup

In [1]:
# Install dependencies (run once)
!pip install -q requests python-dotenv

import os, sys

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AuraVision'
    print('Running in Colab — project root:', PROJECT_ROOT)
except ImportError:
    PROJECT_ROOT = os.path.abspath('..')
    print('Running locally — project root:', PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Working directory:', os.getcwd())

Running locally — project root: c:\Users\Asus\Downloads\AuraVision\AuraVision
Working directory: c:\Users\Asus\Downloads\AuraVision\AuraVision


## Configure API Key

Your Pexels API key is stored in `.env` — **never hardcoded**.  
If the file does not exist yet, create it:
```
PEXELS_API_KEY=your_actual_key_here
```

In [2]:
from dotenv import load_dotenv
load_dotenv()   # loads .env from the project root

api_key = os.getenv('PEXELS_API_KEY')
if not api_key or api_key == 'your_pexels_api_key_here':
    raise EnvironmentError(
        'PEXELS_API_KEY not set!\n'
        'Add it to AuraVision/.env:\n'
        '  PEXELS_API_KEY=your_actual_key_here'
    )
print('✅ API key loaded successfully.')

✅ API key loaded successfully.


## Download Images Images

Downloads images for **Children**, **Adults**, and **Seniors** into `extracted_data/`.  
~40 images per query × 10-12 queries per class ≈ 400–480 images per class.

In [3]:
from data_extraction import download_images
download_images()


📥 Downloading TRAINING images → extracted_data/

  ✓ 'children' → 40 images saved to 'Children/'
  ✓ 'children at school' → 40 images saved to 'Children/'
  ✓ 'playground children' → 40 images saved to 'Children/'
  ✓ 'children playing' → 40 images saved to 'Children/'
  ✓ 'kindergarten' → 40 images saved to 'Children/'
  ✓ 'youth sports' → 40 images saved to 'Children/'
  ✓ 'Elementary school assembly' → 40 images saved to 'Children/'
  ✓ 'Family day at zoo' → 40 images saved to 'Children/'
  ✓ 'Kids birthday party background' → 40 images saved to 'Children/'
  ✓ 'Youth soccer match crowd' → 40 images saved to 'Children/'
  ✓ 'business women' → 40 images saved to 'Adults/'
  ✓ 'business men' → 40 images saved to 'Adults/'
  ✓ 'university students' → 40 images saved to 'Adults/'
  ✓ 'new married couples' → 40 images saved to 'Adults/'
  ✓ 'adults working' → 40 images saved to 'Adults/'
  ✓ 'music festival adults' → 40 images saved to 'Adults/'
  ✓ 'adults in gym' → 40 images saved to 

## Split Into Train / Test

Shuffles `crowd_dataset/` and copies 80% → `dataset/train/`, 20% → `dataset/test/`.  
The split is reproducible (seed = 42).

In [4]:
from split_data import split_dataset
split_dataset(
    source_dir='extracted_data',
    output_dir='dataset',
    split_ratio=0.8,
)

  Adults        train:  416  |  test:  104
  Children      train:  320  |  test:   80
  Seniors       train:  288  |  test:   72

  ── Totals ──  train: 1024  |  test: 256

✅ Split complete → dataset/


## Verify Dataset

Count images in each split to confirm everything looks right.

In [7]:
for subset in ('train', 'test'):
    print(f'\n── dataset/{subset}/')
    base = os.path.join('dataset', subset)
    for cls in sorted(os.listdir(base)):
        cls_path = os.path.join(base, cls)
        if os.path.isdir(cls_path):
            count = len([f for f in os.listdir(cls_path) if f.endswith('.jpg')])
            print(f'  {cls:<12} {count} images')


── dataset/train/
  Adults       416 images
  Children     320 images
  Seniors      288 images

── dataset/test/
  Adults       104 images
  Children     80 images
  Seniors      72 images
